# Hierarchical Multi-Agent Deep Research & Report Planner
This notebook demonstrates a hierarchical multi-agent architecture using **pure LangChain (create_agent)** and **Google Gemini** for deep research and structured report generation.

### System Architecture:
```
                      ┌──────────────────┐
                      │   User Request   │
                      └────────┬─────────┘
                               │
                               ▼
                      ┌──────────────────┐
                      │    Main Agent    │
                      │ (Research Coord) │
                      └────────┬─────────┘
                               │
          ┌────────────────────┼────────────────────────┐
          │                    │                        │
          ▼                    ▼                        ▼
┌──────────────────┐  ┌──────────────────┐┌──────────────────┐  ┌──────────────────┐
│    Subagent A    │  │    Subagent B    ││    Subagent C    │  │    Subagent D    │
│ (Section Writer) │  │  (Intro Writer)  ││ (Conclusion Wr)  │  │ (Compiler)       │
└──────────────────┘  └──────────────────┘└──────────────────┘  └──────────────────┘
```

## 1. Setup & Environment Variables
Set up your API keys from `.env` or interactively.

In [14]:
import os
from getpass import getpass
from dotenv import load_dotenv

# Load keys from root directory .env if present
load_dotenv(dotenv_path="../.env")

if "GEMINI_API_KEY" not in os.environ:
    if "GOOGLE_API_KEY" in os.environ:
        os.environ["GEMINI_API_KEY"] = os.environ["GOOGLE_API_KEY"]
    else:
        os.environ["GEMINI_API_KEY"] = getpass("Enter your GEMINI API Key: ")

if "TAVILY_API_KEY" not in os.environ:
    os.environ["TAVILY_API_KEY"] = getpass("Enter your Tavily API Key: ")

## 2. Research Tools
Define the Tavily search tool that our specialized Section Researcher subagent will use.

In [15]:
from langchain.tools import tool
from langchain_community.utilities.tavily_search import TavilySearchAPIWrapper

tavily_wrapper = TavilySearchAPIWrapper()

@tool
def search_web(query: str) -> str:
    """Searches the web for up-to-date information on a query."""
    try:
        results = tavily_wrapper.results(query, max_results=3)
        if not results:
            return f"No results found for query: '{query}'."
        
        output = f"Web Search Results for '{query}':\n"
        for i, res in enumerate(results, 1):
            output += f"Source {i}: {res.get('title', 'Untitled')}\n"
            output += f"URL: {res.get('url', 'N/A')}\n"
            output += f"Content: {res.get('content', '')}\n\n"
        return output
    except Exception as e:
        return f"Web search encountered an error: {str(e)}"

## 3. Subagent Instantiation
Instantiate the specialized subagents using `create_agent` from LangChain.

In [16]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0)

# 1. Section Researcher & Writer
section_writer_agent = create_agent(
    model=llm,
    tools=[search_web],
    system_prompt=(
        "You are a specialized Section Researcher & Writer Agent. Your job is to research "
        "and write a detailed section of a report in Markdown based on a given topic, section name, "
        "and description. Use the search_web tool to gather information. "
        "Write a highly technical, factual section of about 150-200 words. "
        "Start with a bold key insight, use short paragraphs, and list your sources at the end."
    )
)

# 2. Intro Writer
intro_writer_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=(
        "You are a specialized Introduction Writer Agent. Your job is to read "
        "completed body sections of a report and write a compelling Introduction "
        "(50-100 words, setting context). Do not call any tools, write directly based on the provided section drafts."
    )
)

# 3. Conclusion Writer
conclusion_writer_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=(
        "You are a specialized Conclusion Writer Agent. Your job is to read "
        "completed body sections of a report and write a structured Conclusion "
        "(100-150 words, synthesizing key findings and ending with next steps). "
        "Include a structured Markdown table summarizing the insights. Do not call any tools."
    )
)

# 4. Report Compiler
report_compiler_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=(
        "You are a specialized Report Compiler Agent. Your job is to assemble all sections "
        "(Introduction, Body Sections, and Conclusion) into a single cohesive Markdown report. "
        "Ensure consistent heading hierarchies, verify smooth transitions, format sources "
        "and ensure all special characters like '$' are escaped as '\\$' for correct rendering."
    )
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


## 4. Wrapping Subagents as Tools
To allow the main coordinator agent to invoke subagents, we wrap each subagent's execution pipeline in a LangChain `@tool`. We ensure `RunnableConfig` is propagated so that all subagent callbacks bubble up to the main stream.

In [17]:
from langchain_core.runnables import RunnableConfig
import datetime

@tool("Section_Researcher_Writer", description="Researches and writes a single section of a report. Input should be a query containing the section name, description, and overall topic.")
def call_section_writer(query: str, config: RunnableConfig) -> str:
    """Call the section researcher subagent."""
    now = datetime.datetime.now().strftime("%H:%M:%S.%f")[:-3]
    print(f"\n[{now}] >>> ENTER: Section_Researcher_Writer (Query: {query[:60]}...)")
    result = section_writer_agent.invoke({'messages': [{'role': 'user', 'content': query}]}, config)
    now = datetime.datetime.now().strftime("%H:%M:%S.%f")[:-3]
    print(f"\n[{now}] <<< EXIT: Section_Researcher_Writer finished")
    return result['messages'][-1].content

@tool("Intro_Writer", description="Drafts the Introduction section. Input must include the overall topic and all completed body sections as context.")
def call_intro_writer(query: str, config: RunnableConfig) -> str:
    """Call the intro subagent."""
    now = datetime.datetime.now().strftime("%H:%M:%S.%f")[:-3]
    print(f"\n[{now}] >>> ENTER: Intro_Writer")
    result = intro_writer_agent.invoke({'messages': [{'role': 'user', 'content': query}]}, config)
    now = datetime.datetime.now().strftime("%H:%M:%S.%f")[:-3]
    print(f"\n[{now}] <<< EXIT: Intro_Writer finished")
    return result['messages'][-1].content

@tool("Conclusion_Writer", description="Drafts the Conclusion section. Input must include the overall topic and all completed body sections as context.")
def call_conclusion_writer(query: str, config: RunnableConfig) -> str:
    """Call the conclusion subagent."""
    now = datetime.datetime.now().strftime("%H:%M:%S.%f")[:-3]
    print(f"\n[{now}] >>> ENTER: Conclusion_Writer")
    result = conclusion_writer_agent.invoke({'messages': [{'role': 'user', 'content': query}]}, config)
    now = datetime.datetime.now().strftime("%H:%M:%S.%f")[:-3]
    print(f"\n[{now}] <<< EXIT: Conclusion_Writer finished")
    return result['messages'][-1].content

@tool("Report_Compiler", description="Assembles and compiles the final report structure. Input must contain the introduction, all body sections, and the conclusion to format and combine.")
def call_report_compiler(query: str, config: RunnableConfig) -> str:
    """Call the report compiler subagent."""
    now = datetime.datetime.now().strftime("%H:%M:%S.%f")[:-3]
    print(f"\n[{now}] >>> ENTER: Report_Compiler")
    result = report_compiler_agent.invoke({'messages': [{'role': 'user', 'content': query}]}, config)
    now = datetime.datetime.now().strftime("%H:%M:%S.%f")[:-3]
    print(f"\n[{now}] <<< EXIT: Report_Compiler finished")
    return result['messages'][-1].content

main_agent_tools = [call_section_writer, call_intro_writer, call_conclusion_writer, call_report_compiler]

## 5. Main Coordinator Agent
Set up the Main Agent (Deep Research Planner) with detailed system instructions guiding it through planning, delegation, and final synthesis. To achieve parallelization, we instruct the coordinator to trigger multiple tool calls concurrently in a single step.

In [18]:
main_agent_prompt = (
    "You are the Deep Research Planner, a main coordinator agent. "
    "Your goal is to coordinate a deep research report generation on a topic requested by the user. "
    "\n\n"
    "Available subagents (exposed as tools):\n"
    "1. Section_Researcher_Writer: Researches and writes a single section of the report.\n"
    "2. Intro_Writer: Drafts the Introduction using the body sections.\n"
    "3. Conclusion_Writer: Drafts the Conclusion using the body sections.\n"
    "4. Report_Compiler: Combines, formats, and compiles all sections into a single markdown report.\n"
    "\n\n"
    "Step-by-step workflow:\n"
    "a. Determine the structure of the report (e.g., plan 2-3 main body sections based on the user's topic).\n"
    "b. Call Section_Researcher_Writer FOR ALL PLANNED SECTIONS IN PARALLEL (trigger multiple tool calls in a single turn/step) to write all body sections concurrently.\n"
    "c. Once the body sections are written, call BOTH Intro_Writer and Conclusion_Writer IN PARALLEL (trigger tool calls for both in a single turn/step) to write the introduction and conclusion concurrently.\n"
    "d. Call Report_Compiler with the Intro, the drafted sections, and the Conclusion to build the final report.\n"
    "e. Output the final compiled report to the user.\n\n"
    "CRITICAL: ALWAYS invoke parallel tool calls together in the same step to achieve concurrency. Do not call them one by one."
)

main_agent = create_agent(
    model=llm,
    tools=main_agent_tools,
    system_prompt=main_agent_prompt
)

## 6. Execution Run
Let's test the entire multi-agent system with a request. Watch it plan, delegate in parallel, and compile!

In [19]:
user_request = (
    "I want a research report on 'LangGraph State Management'. "
    "Include sections on State Persistence (memory) and Human-in-the-loop interaction."
)

print("=== STARTING DEEP RESEARCH COORDINATOR AGENT ===\n")
result = main_agent.invoke({"messages": [{"role": "user", "content": user_request}]})
print("\n=== FINAL REPORT RESPONSE ===\n")
print(result["messages"][-1].content[0]['text'] if isinstance(result["messages"][-1].content, list) else result["messages"][-1].content)

=== STARTING DEEP RESEARCH COORDINATOR AGENT ===


[14:03:33.410] >>> ENTER: Section_Researcher_Writer (Query: Section: State Persistence in LangGraph. Topic: LangGraph St...)

[14:03:33.418] >>> ENTER: Section_Researcher_Writer (Query: Section: Human-in-the-loop Interaction in LangGraph. Topic: ...)

[14:03:38.213] <<< EXIT: Section_Researcher_Writer finished

[14:03:39.235] <<< EXIT: Section_Researcher_Writer finished

[14:03:40.322] >>> ENTER: Intro_Writer

[14:03:40.325] >>> ENTER: Conclusion_Writer

[14:03:41.961] <<< EXIT: Intro_Writer finished

[14:03:42.373] <<< EXIT: Conclusion_Writer finished

[14:03:43.361] >>> ENTER: Report_Compiler

[14:03:47.287] <<< EXIT: Report_Compiler finished

=== FINAL REPORT RESPONSE ===

# Report: LangGraph State Management

## 1. Introduction
In the development of complex, multi-agent AI systems, managing the flow of information and the evolution of data is critical. LangGraph provides a robust framework for building stateful, multi-actor applica

In [20]:
print(.)

SyntaxError: invalid syntax (1297590807.py, line 1)

## 7. TRUE STREAMING with astream_events
This cell implements real-time streaming of all agents and subagents using `astream_events` (version `"v2"`). You will see intermediate tool calls executing in parallel, and model tokens appearing as they are generated, colored by active agent.

In [ ]:
import time
import asyncio
import nest_asyncio
from rich.console import Console
from rich.panel import Panel
from langchain_core.messages import HumanMessage

nest_asyncio.apply()
console = Console()

async def true_stream_agent_async(agent, user_request: str):
    console.print("\n" + "="*100, style="bold cyan")
    console.print("🚀 DEEP RESEARCH COORDINATOR (TRUE REAL-TIME STREAMING)", style="bold magenta")
    console.print("="*100, style="bold cyan")
    console.print(f"\n📋 [bold yellow]User Request:[/bold yellow]")
    console.print(Panel(user_request, style="blue", expand=False))
    console.print("\n[bold magenta]🤖 Starting TRUE STREAMING Execution (using astream_events v2)...[/bold magenta]")
    console.print("[bold cyan]" + "─" * 100 + "[/bold cyan]\n")
    
    active_tools = set()
    start_time = time.time()
    tool_count = 0
    
    try:
        async for event in agent.astream_events(
            {"messages": [HumanMessage(content=user_request)]},
            version="v2"
        ):
            event_type = event.get("event")
            name = event.get("name")
            
            # Handle Tool Calls
            if event_type == "on_tool_start":
                active_tools.add(name)
                tool_count += 1
                inputs = event.get("data", {}).get("input")
                console.print(f"\n\n[bold magenta]🔧 [Tool Start] {name}[/bold magenta]")
                console.print(f"  [dim]Input: {inputs}[/dim]\n")
                
            elif event_type == "on_tool_end":
                if name in active_tools:
                    active_tools.remove(name)
                output = event.get("data", {}).get("output")
                if hasattr(output, 'content'):
                    output_str = str(output.content)
                elif isinstance(output, list) and len(output) > 0 and isinstance(output[0], dict) and 'text' in output[0]:
                    output_str = output[0]['text']
                else:
                    output_str = str(output)
                console.print(f"\n[bold green]✓ [Tool End] {name} Result:[/bold green]")
                console.print(Panel(output_str.strip(), style="green", expand=False))
                console.print()
                
            # Handle LLM Token Streaming
            elif event_type == "on_chat_model_stream":
                chunk = event.get("data", {}).get("chunk")
                content = chunk.content if hasattr(chunk, 'content') else str(chunk)
                
                # Color-code based on which agent is generating the response
                if "Section_Researcher_Writer" in active_tools:
                    console.print(content, style="green", end="")
                elif "Intro_Writer" in active_tools:
                    console.print(content, style="yellow", end="")
                elif "Conclusion_Writer" in active_tools:
                    console.print(content, style="red", end="")
                elif "Report_Compiler" in active_tools:
                    console.print(content, style="blue", end="")
                else:
                    # Main coordinator agent (Deep Research Planner)
                    console.print(content, style="cyan", end="")
                    
        total_time = time.time() - start_time
        console.print("\n" + "="*100, style="bold cyan")
        console.print("✨ EXECUTION COMPLETE", justify="center", style="bold green")
        console.print("="*100, style="bold cyan")
        console.print(f"\n[bold yellow]📊 Stream Statistics:[/bold yellow]")
        console.print(f"  • Total execution time: {total_time:.2f}s")
        console.print(f"  • Tool calls executed: {tool_count}\n")
        
    except Exception as e:
        console.print(f"\n[bold red]❌ ERROR:[/bold red] {str(e)}")
        import traceback
        traceback.print_exc()

# Execute TRUE STREAMING
user_request_stream = (
    "I want a research report on 'LangGraph State Management'. "
    "Include sections on State Persistence (memory) and Human-in-the-loop interaction."
)
asyncio.run(true_stream_agent_async(main_agent, user_request_stream))